# Laboratory 7: Quantum Optics — Single Photons, Interference, and Entangled Light

Light arrives one photon at a time. The interactive lab guide's optical bench showed you
four experiments; this notebook rebuilds each of them as a **quantum circuit** and
reproduces the same numbers, seeded end to end. The questions driving the four parts:

1. **One photon at a beam splitter** — what does it mean that light comes in photons, and
   what happens when one *indivisible* photon hits a half-silvered mirror? (Preset A)
2. **The Mach–Zehnder interferometer** — if the photon went one way *or* the other, the
   detector counts could not depend on the path it didn't take. They do. What now?
   (Presets B1/B2)
3. **Polarization** — two crossed filters block everything; insert a third *between* them
   and light comes through. Why does adding an absorber *increase* transmission? (Preset C)
4. **Entangled pairs** *(optional)* — two photons, each individually a perfect coin toss,
   always agree when asked the same question. What kind of correlation is that? (Preset D)

A single photon carries a qubit two ways — its **path** (dual-rail: $|A\rangle = |0\rangle$,
$|B\rangle = |1\rangle$) and its **polarization** ($|H\rangle = |0\rangle$,
$|V\rangle = |1\rangle$) — so every optical element below is a one- or two-qubit gate, and
every measured number is a Born-rule statistic you can check against a closed formula.

> ### Convention note — bit ordering (it genuinely matters in this lab)
>
> This lab series reports every quantum state in the **lab convention (big-endian)**:
> in $|q_0 q_1 \dots q_{n-1}\rangle$, qubit $0$ is the **leftmost** bit. **Qiskit uses the
> opposite (little-endian) order**, with qubit $0$ rightmost in returned bitstrings.
>
> Unlike Labs 5 and 6, whose registers were single qubits, this notebook uses **two-qubit
> registers** — the which-path marker (Section 3) and the photon pairs (Section 5) — so the
> conversion is a genuine bit reversal, not the identity:
>
> | Qiskit string | lab string |
> |---|---|
> | `00` | `00` |
> | `01` | `10` |
> | `10` | `01` |
> | `11` | `11` |
>
> Every displayed multi-qubit bitstring in this notebook passes through the series'
> converter `qiskit_to_lab()` (defined and round-trip-checked in the setup cell).

### Environment Setup

In [ ]:
"""
Environment Setup Module.
Run this cell first to install all required libraries.
"""
%pip install -q qiskit qiskit-aer matplotlib

import numpy as np
import matplotlib.pyplot as plt

import qiskit
import qiskit_aer
from qiskit import ClassicalRegister, QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

# All randomness in this notebook is seeded, so every run reproduces these numbers.
SEED = 42


def qiskit_to_lab(bitstring: str) -> str:
    """Convert a Qiskit (little-endian) bitstring to lab (big-endian) convention.

    A genuine bit reversal in this notebook: two-qubit registers appear in
    Sections 3 and 5, where '01' and '10' swap meaning between the conventions.
    """
    return bitstring[::-1]


def lab_to_qiskit(bitstring: str) -> str:
    """Convert a lab (big-endian) bitstring to Qiskit (little-endian) convention."""
    return bitstring[::-1]


def sample_counts(qc: QuantumCircuit, shots: int, seed: int) -> dict:
    """Run a measured circuit on the seeded Aer simulator -> counts dict."""
    sim = AerSimulator(seed_simulator=seed)
    return sim.run(transpile(qc, sim), shots=shots).result().get_counts()


# The two-qubit conversion table, checked (this is the mapping the lab guide states):
print("bit-ordering conversion (2-qubit registers):")
for q, lab in [("00", "00"), ("01", "10"), ("10", "01"), ("11", "11")]:
    assert qiskit_to_lab(q) == lab, "conversion table violated"
    assert lab_to_qiskit(qiskit_to_lab(q)) == q, "round trip must be the identity"
    print(f"  Qiskit '{q}'  ->  lab '{lab}'")

print("\nqiskit    ", qiskit.__version__)
print("qiskit-aer", qiskit_aer.__version__)
print("numpy     ", np.__version__)

---
## 1 · One Photon at a Beam Splitter

**The question first:** *what does it actually mean that light comes in photons — and what
happens when one indivisible photon hits a half-silvered mirror?*

**Why this step:** the beam splitter is the elementary gate of the optical bench, and its
statistics carry the first shock of the lab: the photon's state *splits*, the photon never
does.

The lab's binding beam-splitter convention is the symmetric 50/50 unitary with phase $i$ on
reflection,
$$B = \frac{1}{\sqrt 2}\begin{pmatrix}1 & i\\ i & 1\end{pmatrix},$$
which as a circuit gate is exactly `rx(-pi/2)` — we *pin this entrywise* before using it.

**Boundary note:** $B$ is the *physical evolution* of the optical circuit, never a
measurement basis change. All measurement basis changes in this notebook (Procedure 2) use
the canonical rotation $U_X = H$ or the declared analyzer family `ry(-2*theta)`
(Section 4).

Detection at the two output ports is a path-basis measurement — **Procedure 1**,
destructive. One measured path qubit yields exactly one classical bit per shot:
the "both detectors click" event is not a rare outcome, it is **not in the outcome set**.

In [ ]:
"""
Part A - single photon at a 50/50 beam splitter.

The beam splitter B = (1/sqrt2)[[1, i], [i, 1]] (i on reflection) is exactly
the gate rx(-pi/2); the convention is pinned entrywise before use.
Detection = path-basis measurement (Procedure 1, destructive).
"""
B_MATRIX = np.array([[1, 1j], [1j, 1]], dtype=complex) / np.sqrt(2)

qc_bs = QuantumCircuit(1)
qc_bs.rx(-np.pi / 2, 0)          # the beam splitter as a circuit gate
defect = np.max(np.abs(Operator(qc_bs).data - B_MATRIX))
assert defect < 1e-12, "rx(-pi/2) must equal the binding beam-splitter matrix"
print(f"convention pinned: max |rx(-pi/2) - B| = {defect:.2e}")

sv = Statevector(qc_bs)          # B|A>, input |A> = |0>
print(f"\nB|A> amplitudes: [{sv[0]:.4f}, {sv[1]:.4f}]   (theory: [1/sqrt2, i/sqrt2])")
probs = sv.probabilities()
assert abs(probs[0] - 0.5) < 1e-12 and abs(probs[1] - 0.5) < 1e-12
print(f"P(port A), P(port B) = {probs[0]:.12f}, {probs[1]:.12f}   (theory: 0.5, 0.5)")

# Seeded sampling: 10 000 single photons
qc = qc_bs.copy()
qc.measure_all()
shots = 10_000
counts = sample_counts(qc, shots, SEED)
frac_b = counts.get('1', 0) / shots
sigma = np.sqrt(0.25 / shots)                       # sqrt(p(1-p)/n) at p = 1/2
assert abs(frac_b - 0.5) < 4 * sigma, "port-B fraction must sit within 4 sigma of 1/2"
print(f"\nsampled ({shots} photons, seed {SEED}):")
print(f"  port A: {counts.get('0', 0):5d}   port B: {counts.get('1', 0):5d}"
      f"   fraction B = {frac_b:.4f}  (theory 0.5000 +- {4*sigma:.4f} at 4 sigma)")

# Anti-correlation, structural: one path qubit -> exactly one outcome bit per shot.
# There is no 'both detectors' key for a coincidence to accumulate in.
assert set(counts) <= {'0', '1'}, "outcome set is exactly {port A, port B}"
assert sum(counts.values()) == shots, "every photon produced exactly one click"
print(f"  coincidence events (both detectors in one trial): 0 of {shots} - structural, exact")

---
## 2 · The Mach–Zehnder Interferometer

**The question first:** *if the photon went one way or the other, the detector counts could
not depend on the path it didn't take. They do. What now?*

On the bench (guide, Preset B1) the interferometer is $B\cdot\Phi(\varphi)\cdot B$ with
$\Phi(\varphi) = \mathrm{diag}(1, e^{i\varphi})$. The natural *circuit* form is
$H\cdot P(\varphi)\cdot H$. **These are not the same matrix — and we will not pretend they
are.** The exact relation, with $S = \mathrm{diag}(1, i)$:

$$B = S\,H\,S
\qquad\Longrightarrow\qquad
B\,\Phi(\varphi)\,B \;=\; S\,\bigl[H\,P(\varphi + \pi)\,H\bigr]\,S .$$

Reading off what is *observable* in a Z-measurement: the right $S$ acts trivially on the
input $|0\rangle$; the left $S$ contributes only Z-diagonal phases, which Z-measurement
probabilities cannot see. What survives is the shift $\varphi \to \varphi + \pi$ —
equivalently, **a swap of the two output labels**. On the bench, the bright-at-$\varphi=0$
detector D1 sits at output port B (path state $|1\rangle$); in the $H$-circuit the
bright-at-$\varphi=0$ outcome is `'0'`.

**Notebook convention, fixed here once:** D1 $\equiv$ the bright-at-$\varphi = 0$ output
$\equiv$ outcome `'0'` of the $H$-circuit. With that identification,
$$P(D1) = \cos^2(\varphi/2), \qquad P(D2) = \sin^2(\varphi/2)$$
in both representations, and every fringe number below matches the bench. The next cell
proves each claim numerically instead of asking you to trust the algebra.

In [ ]:
"""
The correspondence B.Phi(phi).B  <->  H.P(phi).H, demonstrated numerically.

(1) B = rx(-pi/2) = S.H.S                          -- exact gate identities;
(2) B.Phi(phi).B is NOT element-wise H.P(phi).H    -- max entry difference ~ 1;
(3) B.Phi(phi).B = S.H.P(phi+pi).H.S               -- exact, to machine precision;
(4) Z-measurement distributions: identical up to the output-label swap.
"""
S_M = np.diag([1, 1j]).astype(complex)
H_M = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
PHASE = lambda phi: np.diag([1, np.exp(1j * phi)]).astype(complex)

# (1) exact gate identities
d_shs = np.max(np.abs(S_M @ H_M @ S_M - B_MATRIX))
assert d_shs < 1e-12, "B must equal S.H.S"
print(f"(1) max |S.H.S - B|                        = {d_shs:.2e}   (exact identity)")

phi_star = np.pi / 3
bench   = B_MATRIX @ PHASE(phi_star) @ B_MATRIX      # the bench pipeline
circuit = H_M @ PHASE(phi_star) @ H_M                # the H-based circuit

# (2) the two forms genuinely differ element-wise
d_naive = np.max(np.abs(bench - circuit))
assert d_naive > 0.5, "the difference is not small - it must not be glossed over"
print(f"(2) max |B.Phi.B - H.P.H|  at phi = pi/3   = {d_naive:.4f}   (NOT the same matrix)")

# (3) the exact identity, phase shifted by pi and conjugated by S
ident = S_M @ H_M @ PHASE(phi_star + np.pi) @ H_M @ S_M
d_exact = np.max(np.abs(bench - ident))
assert d_exact < 1e-12, "B.Phi(phi).B must equal S.H.P(phi+pi).H.S"
print(f"(3) max |B.Phi.B - S.H.P(phi+pi).H.S|      = {d_exact:.2e}   (exact identity)")

# (4) Z-measurement statistics: same fringes, output labels swapped
ket0 = np.array([1, 0], dtype=complex)
p_bench = np.abs(bench @ ket0) ** 2
p_circ  = np.abs(circuit @ ket0) ** 2
print(f"(4) bench   outcome probabilities: [{p_bench[0]:.6f}, {p_bench[1]:.6f}]")
print(f"    circuit outcome probabilities: [{p_circ[0]:.6f}, {p_circ[1]:.6f}]")
assert abs(p_bench[0] - p_circ[1]) < 1e-12 and abs(p_bench[1] - p_circ[0]) < 1e-12
print("    -> identical distributions with the two output labels SWAPPED, as derived.")
print("       Notebook convention: D1 = bright-at-phi=0 = outcome '0' of the H-circuit.")

In [ ]:
"""
Part B - Mach-Zehnder fringes: P(D1) = cos^2(phi/2).

Statevector sweep (exact, tol 1e-12) + seeded sampling at the anchor phases.
D1 = bright-at-phi=0 output = outcome '0' (Section 2 convention).
"""
def mz_circuit(phi: float) -> QuantumCircuit:
    """H . P(phi) . H on the path qubit - the Mach-Zehnder as a circuit."""
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.p(phi, 0)
    qc.h(0)
    return qc


# Exact sweep: statevector P(D1) against the closed form
phis = np.linspace(0.0, 2 * np.pi, 25)
defect = max(abs(Statevector(mz_circuit(p)).probabilities()[0] - np.cos(p / 2) ** 2)
             for p in phis)
assert defect < 1e-12, "statevector fringe must match cos^2(phi/2)"
print(f"statevector sweep, 25 phases: max |P(D1) - cos^2(phi/2)| = {defect:.2e}")

# Seeded sampling at the three anchor phases.
# phi = 0: the dark amplitude (1 - e^{i0})/2 is 0 EXACTLY, in float arithmetic too.
# phi = pi: the dark probability is ~ 3.7e-33 - one float ulp from exact zero.
shots = 10_000
for phi, label, seed, note in [
        (0.0,       "phi = 0   ", SEED + 1, "structural zero (exact)"),
        (np.pi / 3, "phi = pi/3", SEED + 2, ""),
        (np.pi,     "phi = pi  ", SEED + 3, "dark P ~ 3.7e-33, one ulp from 0")]:
    qc = mz_circuit(phi)
    qc.measure_all()
    counts = sample_counts(qc, shots, seed)
    frac_d1 = counts.get('0', 0) / shots
    theory = np.cos(phi / 2) ** 2
    if note:                                        # structural anchors
        dark = counts.get('1' if phi == 0.0 else '0', 0)
        assert dark == 0, "the dark port must never fire at a fringe null"
        print(f"{label}: D1 fraction = {frac_d1:.4f}   theory {theory:.4f}   ({note})")
    else:
        sigma = np.sqrt(theory * (1 - theory) / shots)
        assert abs(frac_d1 - theory) < 4 * sigma
        print(f"{label}: D1 fraction = {frac_d1:.4f}   theory {theory:.4f}"
              f"   (+- {4*sigma:.4f} at 4 sigma)")

# Sampled fringe (echoes the guide's Preset B1 'Sweep phi')
sweep_phis = np.linspace(0.0, 2 * np.pi, 13)
dots = []
for i, phi in enumerate(sweep_phis):
    qc = mz_circuit(phi)
    qc.measure_all()
    dots.append(sample_counts(qc, 2000, SEED + 100 + i).get('0', 0) / 2000)

grid = np.linspace(0, 2 * np.pi, 200)
plt.figure(figsize=(7, 3.2))
plt.plot(grid, np.cos(grid / 2) ** 2, label="theory $\\cos^2(\\varphi/2)$")
plt.plot(sweep_phis, dots, "o", label="sampled (2000 shots/point)")
plt.xlabel("phase $\\varphi$ [rad]"); plt.ylabel("P(D1)")
plt.title("Mach-Zehnder fringe (compare the guide's Preset B1 sweep)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 3 · Which-Path Marking: Complementarity, $\mathcal{V} = s$

**Why this step:** the fringes above exist because the two paths are *indistinguishable*.
Record which path the photon took and the fringes must die — and here is the shock, stated
openly: **nobody needs to read the record.** Its existence is enough.

Model: a **marker qubit** is coupled to the path between the beam splitters. Path A leaves
the marker in $|M_A\rangle = |0\rangle$; path B rotates it to
$|M_B\rangle = s\,|0\rangle + \sqrt{1-s^2}\,|1\rangle$, so the overlap
$\langle M_A|M_B\rangle = s$ is dialed directly: `cry(2*arccos(s))` controlled by the path.
$s = 0$ is a CNOT (perfect record); $s = 1$ leaves no record. The claim to verify is the
complementarity identity — fringe **visibility equals marker indistinguishability**,
$\mathcal{V} = s$.

**Wiring and bit order (this is where the convention note bites):** the path is Qiskit
qubit 0 and the marker is Qiskit qubit 1. Qiskit prints qubit 0 *rightmost*; the lab
convention writes $|\text{path}\;\text{marker}\rangle$ with the path *leftmost*. Every
displayed two-bit string below is therefore converted with `qiskit_to_lab()` — e.g. the
amplitude Qiskit stores at bitstring `'01'` belongs to the lab state $|10\rangle$ (photon
in path B, marker untouched).

In the visibility experiment we measure **only the path qubit**. The marker is never read.

In [ ]:
"""
Perfect marking (s = 0): the record, in amplitudes and in counts.

Displayed two-qubit strings are converted with qiskit_to_lab (path leftmost).
The post-BS1 state (marker still |0>) makes the conversion visibly non-trivial:
its second amplitude sits at Qiskit '01' but is lab |10>.
"""
# After BS1 only (H on the path qubit; marker untouched)
qc1 = QuantumCircuit(2)          # qubit 0 = path, qubit 1 = marker
qc1.h(0)
sv1 = Statevector(qc1)
print("state after BS1, before marking - amplitude table:")
print("  index | Qiskit string | lab |path marker>  | amplitude")
for idx in range(4):
    q_str = format(idx, '02b')
    lab_str = qiskit_to_lab(q_str)
    print(f"    {idx}   |      '{q_str}'     |     |{lab_str}>       "
          f"| {sv1[idx]: .4f}")
r2 = 1 / np.sqrt(2)
assert abs(sv1[1] - r2) < 1e-12, "the 1/sqrt2 amplitude sits at Qiskit '01' = lab |10>"
assert sv1[2] == 0 and sv1[3] == 0, "marker still |0>: lab |01>, |11> amplitudes exactly 0"
print("  -> lab reading: (|00> + |10>)/sqrt2 - photon split over paths, marker untouched.")
print("     (A little-endian misreading would call it (|00> + |01>)/sqrt2 - wrong state.)")

# Now mark perfectly: CNOT, path controls marker
qc2 = QuantumCircuit(2)
qc2.h(0)
qc2.cx(0, 1)
sv2 = Statevector(qc2)
assert sv2[1] == 0 and sv2[2] == 0, "structural zeros: only |00> and |11> are populated"
print(f"\nafter marking (CNOT): lab (|00> + |11>)/sqrt2, amplitudes "
      f"[{sv2[0]:.4f}, {sv2[3]:.4f}] on |00>, |11>")

qc2.measure_all()
shots = 10_000
counts = sample_counts(qc2, shots, SEED + 4)
print(f"\nsampled counts ({shots} shots, seed {SEED + 4}):")
print("  Qiskit key -> lab key | count")
for key in sorted(counts):
    print(f"      '{key}'   ->  '{qiskit_to_lab(key)}'  | {counts[key]:5d}")
lab_keys = {qiskit_to_lab(k) for k in counts}
assert lab_keys <= {'00', '11'}, "path and marker agree in every single shot - exact"
sigma = np.sqrt(0.25 / shots)
assert abs(counts.get('00', 0) / shots - 0.5) < 4 * sigma
print("  -> mixed keys '01'/'10': 0 shots (structural, exact) - the record is perfect.")

In [ ]:
"""
The complementarity identity: fringe visibility = marker overlap, V = s.

Marker NEVER measured. Statevector fringes exact (tol 1e-12); seeded sampling
at the s = 0.5 extremes. P(D1) = P(path bit = 0) = |sv[0]|^2 + |sv[2]|^2
(Qiskit indices 0 = '00' and 2 = '10' both have path qubit q0 = 0).
"""
def marked_mz(phi: float, s: float) -> QuantumCircuit:
    """MZ with a which-path marker of overlap s (marker = qubit 1, unread)."""
    qc = QuantumCircuit(2)
    qc.h(0)                              # BS1
    qc.cry(2 * np.arccos(s), 0, 1)       # marking: |M_B> = s|0> + sqrt(1-s^2)|1>
    qc.p(phi, 0)                         # phase in arm B
    qc.h(0)                              # BS2
    return qc


def p_d1(phi: float, s: float) -> float:
    sv = Statevector(marked_mz(phi, s))
    return abs(sv[0]) ** 2 + abs(sv[2]) ** 2


# Exact fringes and visibility for five marker settings
print("s      visibility V (statevector)   theory V = s")
phis = np.linspace(0.0, 2 * np.pi, 17)
for s in [0.0, 0.25, 0.5, 0.75, 1.0]:
    defect = max(abs(p_d1(p, s) - (1 + s * np.cos(p)) / 2) for p in phis)
    assert defect < 1e-12, "fringe must match (1 + s cos phi)/2"
    p0, ppi = p_d1(0.0, s), p_d1(np.pi, s)
    vis = (max(p0, ppi) - min(p0, ppi)) / (max(p0, ppi) + min(p0, ppi))
    assert abs(vis - s) < 1e-12, "visibility must equal the marker overlap"
    print(f"{s:.2f}          {vis:.12f}            {s:.2f}")

# Seeded sampling at s = 0.5: the 0.75 / 0.25 extremes of the guide's Preset B2
shots = 10_000
for phi, seed, theory in [(0.0, SEED + 5, 0.75), (np.pi, SEED + 6, 0.25)]:
    qc = marked_mz(phi, 0.5)
    qc.add_register(ClassicalRegister(1, 'c'))
    qc.measure(0, 0)                     # path only - the marker stays unread
    counts = sample_counts(qc, shots, seed)
    frac = counts.get('0', 0) / shots
    sigma = np.sqrt(theory * (1 - theory) / shots)
    assert abs(frac - theory) < 4 * sigma
    print(f"\nsampled s = 0.5, phi = {phi:.2f}: D1 fraction = {frac:.4f}"
          f"   theory {theory:.4f} (+- {4*sigma:.4f} at 4 sigma)")

# Fringe families (the guide's Preset B2 picture)
grid = np.linspace(0, 2 * np.pi, 200)
plt.figure(figsize=(7, 3.2))
for s in [1.0, 0.5, 0.0]:
    plt.plot(grid, (1 + s * np.cos(grid)) / 2, label=f"s = {s}")
plt.xlabel("phase $\\varphi$ [rad]"); plt.ylabel("P(D1)")
plt.title("Which-path marking: visibility = s (marker unread)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("\nThe marker was never measured. Distinguishability alone - the existence of the")
print("record - sets the visibility. This is Lab 6's 'Eve leaves fingerprints', as algebra.")

---
## 4 · Polarization: Quantum Malus, Two Procedures, Three Polarizers

**The question first:** *two crossed filters block everything. Insert a third filter
between them and light comes through. Why does adding an absorber increase transmission?*

Polarization is the photon's second qubit: $|H\rangle = |0\rangle$,
$|V\rangle = |1\rangle$, and the linear-polarization state at angle $\theta$ is
$|\theta\rangle = \cos\theta\,|H\rangle + \sin\theta\,|V\rangle$ — prepared by
`ry(2*theta)`. Measuring in the $\{|\theta\rangle, |\theta^\perp\rangle\}$ basis is
**Procedure 2** with the declared analyzer family `ry(-2*theta)`; at $\theta = 45°$
(the X-basis) the canonical rotation $U_X = H$ is used, and we check the family agrees
with it there — they differ by a Z-diagonal sign that Z-measurement cannot see.

The X-basis measurement at $45°$ exists as **two different devices**:

- **Procedure 1** — an absorptive polarizer film at 45°: probabilities
  $|\langle\pm|\psi\rangle|^2$, passed photon left in $|+\rangle$;
- **Procedure 2** — a half-waveplate at 22.5° (implementing $H$) + polarizing beam
  splitter: *identical probabilities*, post-state $|H\rangle$ or $|V\rangle$.

Same statistics, different post-measurement states — the distinction that decided Lab 6's
naive-resend attack (37.5% vs 25%). Both sides are made runnable below.

In [ ]:
"""
Quantum Malus law: P(pass) = cos^2 theta.

Input |H> = |0>; polarizer at theta = analyzer rotation ry(-2 theta) then
Z-measurement, outcome '0' = pass (Procedure 2 realization of the statistics).
"""
anchors = [(0.0, 1.0), (np.pi / 6, 0.75), (np.pi / 4, 0.5),
           (np.pi / 3, 0.25), (np.pi / 2, 0.0)]

print("theta   P(pass) statevector   theory")
for theta, expected in anchors:
    qc = QuantumCircuit(1)               # input |H>
    qc.ry(-2 * theta, 0)                 # analyzer rotation U_theta
    p_pass = Statevector(qc).probabilities()[0]
    assert abs(p_pass - expected) < 1e-12, "Malus law must hold exactly"
    print(f"{np.degrees(theta):5.1f}   {p_pass:.12f}     {expected}")

# Seeded sampling at the 60-degree anchor (the guide's Preset C loop-back)
shots = 10_000
qc = QuantumCircuit(1)
qc.ry(-2 * np.pi / 3, 0)
qc.measure_all()
counts = sample_counts(qc, shots, SEED + 7)
frac = counts.get('0', 0) / shots
sigma = np.sqrt(0.25 * 0.75 / shots)
assert abs(frac - 0.25) < 4 * sigma
print(f"\nsampled theta = 60 deg: pass fraction = {frac:.4f}"
      f"   theory 0.2500 (+- {4*sigma:.4f} at 4 sigma)")
print("Each photon entirely passes or is entirely absorbed - cos^2 is a Born probability.")

In [ ]:
"""
Procedure 1 vs Procedure 2 at the 45-degree point, runnable.

Probabilities identical (checked to 1e-12 and by sampling); post-measurement
states different (cross-fidelity exactly 1/2). Also: the analyzer family
ry(-pi/2) agrees with canonical U_X = H on Z-probabilities.
"""
theta0 = np.pi / 7
prep = QuantumCircuit(1)
prep.ry(2 * theta0, 0)                   # |theta_0>
psi = Statevector(prep)

# Procedure 1: direct projection onto |+>, |-> (the polarizer film, computed)
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
ket_minus = np.array([1, -1], dtype=complex) / np.sqrt(2)
p1 = np.array([abs(np.vdot(ket_plus, psi.data)) ** 2,
               abs(np.vdot(ket_minus, psi.data)) ** 2])

# Procedure 2: canonical U_X = H, then Z-measure (the HWP@22.5 + PBS device)
qc2 = prep.copy()
qc2.h(0)
p2 = Statevector(qc2).probabilities()
assert np.max(np.abs(p1 - p2)) < 1e-12, "the two procedures' probabilities must agree"
print(f"Procedure 1 (projection): P(+), P(-) = {p1[0]:.12f}, {p1[1]:.12f}")
print(f"Procedure 2 (H then Z):   P(0), P(1) = {p2[0]:.12f}, {p2[1]:.12f}")
print(f"max difference = {np.max(np.abs(p1 - p2)):.2e}  -> identical statistics")

# The declared analyzer family at the X point: ry(-pi/2) vs canonical H
qc_fam = prep.copy()
qc_fam.ry(-np.pi / 2, 0)
p_fam = Statevector(qc_fam).probabilities()
assert np.max(np.abs(p_fam - p2)) < 1e-12, "family and canonical rotation must agree"
print(f"analyzer family ry(-pi/2) vs canonical H: max diff = "
      f"{np.max(np.abs(p_fam - p2)):.2e}  (a Z-diagonal sign, invisible here)")

# Sampled Procedure 2
shots = 10_000
qc2m = qc2.copy()
qc2m.measure_all()
frac = sample_counts(qc2m, shots, SEED + 8).get('0', 0) / shots
sigma = np.sqrt(p2[0] * (1 - p2[0]) / shots)
assert abs(frac - p2[0]) < 4 * sigma
print(f"sampled Procedure 2: P(+) = {frac:.4f}   theory {p2[0]:.4f} "
      f"(+- {4*sigma:.4f} at 4 sigma)")

# Post-measurement states differ: |+> (Procedure 1) vs |0> (Procedure 2)
ket0 = np.array([1, 0], dtype=complex)
fidelity = abs(np.vdot(ket_plus, ket0)) ** 2
assert abs(fidelity - 0.5) < 1e-12
print(f"\npost-state overlap |<+|H>|^2 = {fidelity:.12f}  -> the devices emit")
print("DIFFERENT photons for the same '+' outcome. Identical statistics, different")
print("post-states: exactly the distinction behind Lab 6's 37.5% naive resend.")

In [ ]:
"""
Polarizer cascades with mid-circuit projection: crossed 0, insert 45 -> 1/8.

Unpolarized input: the polarization qubit is entangled with a 'mix' ancilla
(H + CX -> reduced state I/2). Each polarizer at theta_j: rotate into its
frame ry(-2 theta_j), MEASURE (the projection - Procedure 1 collapse), rotate
back ry(+2 theta_j). The photon passes the rack iff every recorded bit is 0.
"""
def cascade_pass_fraction(angles_deg, shots, seed):
    """Fraction of photons passing every polarizer (unpolarized input)."""
    n = len(angles_deg)
    qc = QuantumCircuit(2, n)            # qubit 0 = polarization, qubit 1 = mix
    qc.h(1)
    qc.cx(1, 0)                          # entangle -> polarization is exactly I/2
    for j, deg in enumerate(angles_deg):
        th = np.radians(deg)
        qc.ry(-2 * th, 0)                # into the polarizer frame
        qc.measure(0, j)                 # the projection (pass = 0, absorb = 1)
        qc.ry(2 * th, 0)                 # back to the lab frame (post-state |theta_j>)
    counts = sample_counts(qc, shots, seed)
    passed = counts.get('0' * n, 0)      # all bits 0 = passed every polarizer
    return passed, counts


shots = 10_000
configs = [([0, 90],         0.0,        "crossed 0/90        ", SEED + 9),
           ([0, 45, 90],     0.125,      "insert 45 between   ", SEED + 10),
           ([0, 30, 60, 90], 27 / 128,   "staircase 0/30/60/90", SEED + 11)]

print("configuration          measured   theory")
for angles, theory, label, seed in configs:
    passed, _ = cascade_pass_fraction(angles, shots, seed)
    frac = passed / shots
    if theory == 0.0:
        # (1/2)cos^2(90deg) ~ 1.9e-33: one float ulp from exact zero - never fires.
        assert passed == 0, "crossed polarizers: no photon may pass"
        print(f"{label}   {frac:.4f}     {theory:.4f}   (structural - 0 events)")
    else:
        sigma = np.sqrt(theory * (1 - theory) / shots)
        assert abs(frac - theory) < 4 * sigma
        print(f"{label}   {frac:.4f}     {theory:.4f}   (+- {4*sigma:.4f} at 4 sigma)")

print("\nAdding the 45-degree absorber raised transmission from 0 to 1/8, and the")
print("gentler staircase transmits even more (27/128 ~ 21%): each projection REWRITES")
print("the polarization to its own |theta_j> - the post-measurement state is the")
print("whole explanation. Compare the guide's Preset C.")

---
## 5 · Optional: Entangled Photon Pairs

**The question first:** *two photons, each individually a perfect coin toss, always agree
when asked the same question — at any angle, chosen after they separated. What kind of
correlation is that?*

The pair state is $|\Phi^{+}\rangle = (|HH\rangle + |VV\rangle)/\sqrt 2$, written
**photon 1 leftmost** (lab convention). In the circuit: photon 1 = Qiskit qubit 0,
photon 2 = Qiskit qubit 1, prepared with `h(0); cx(0, 1)` — so, as in Section 3, every
displayed two-bit string is converted with `qiskit_to_lab()`. Each side's analyzer at angle
$\alpha$ (resp. $\beta$) is the rotation `ry(-2*alpha)` before a Z-measurement; outcome
`0` is the "+" port.

Two exact facts collide: each photon alone is *exactly* unpolarized ($P({+}) = 1/2$ at
every angle), yet the correlation is
$$E(\alpha, \beta) = \cos 2\Delta, \qquad \Delta = \alpha - \beta,$$
with **certain agreement at any matched angle**. The best classical wave model (shared
random polarization $\lambda$, Malus response, averaged over $\lambda$) yields
$E_{\text{cl}} = \cos(2\Delta)/2$ — the same shape at **half the amplitude**: 75%
matched-angle agreement against the observed 100%. That factor of 2 is device-verifiable,
and it is the gap the sweep below displays.

In [ ]:
"""
Matched analyzers (alpha = beta = pi/7): individually random, perfectly
correlated. Displayed two-qubit strings converted with qiskit_to_lab
(photon 1 leftmost).
"""
def pair_circuit(alpha: float, beta: float) -> QuantumCircuit:
    """|Phi+> then analyzers at alpha (photon 1 = qubit 0), beta (photon 2)."""
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)                          # (|HH> + |VV>)/sqrt2, photon 1 = qubit 0
    qc.ry(-2 * alpha, 0)                 # analyzer, photon 1
    qc.ry(-2 * beta, 1)                  # analyzer, photon 2
    qc.measure([0, 1], [0, 1])
    return qc


alpha = beta = np.pi / 7
shots = 10_000
counts = sample_counts(pair_circuit(alpha, beta), shots, SEED + 13)

print(f"matched analyzers at {np.degrees(alpha):.1f} deg ({shots} pairs, "
      f"seed {SEED + 13}):")
print("  Qiskit key -> lab key ('+'=0) | count")
lab_counts = {}
for key in sorted(counts):
    lab_key = qiskit_to_lab(key)
    lab_counts[lab_key] = counts[key]
    print(f"      '{key}'   ->   '{lab_key}'        | {counts[key]:5d}")

# Perfect correlation: disagreement outcomes are structurally absent
# (their amplitudes are sin(a)cos(a) - cos(a)sin(a) with IDENTICAL floats = exactly 0)
disagree = lab_counts.get('01', 0) + lab_counts.get('10', 0)
assert disagree == 0, "matched analyzers must agree in every single pair - exact"
print(f"  disagreements ('01' + '10'): {disagree} of {shots} - structural, exact")

# Individually: each side is a fair coin (its analyzer angle notwithstanding)
p1_plus = sum(v for k, v in lab_counts.items() if k[0] == '0') / shots
p2_plus = sum(v for k, v in lab_counts.items() if k[1] == '0') / shots
sigma = np.sqrt(0.25 / shots)
assert abs(p1_plus - 0.5) < 4 * sigma and abs(p2_plus - 0.5) < 4 * sigma
print(f"  one-sided '+' rates: photon 1 = {p1_plus:.4f}, photon 2 = {p2_plus:.4f}"
      f"   (theory 0.5000 +- {4*sigma:.4f})")
print("  -> each photon alone: a perfect coin. Together: locked. Agreement 100%.")

In [ ]:
"""
The correlation curve E(Delta) = cos 2 Delta vs the classical wave's
E_cl = cos(2 Delta)/2 - the factor of 2, measured.

E = P(++) + P(--) - P(+-) - P(-+); statevector exact, sampled dots overlaid.
"""
def pair_probs(alpha: float, beta: float) -> np.ndarray:
    """[P(++), P(-+), P(+-), P(--)] in Statevector index order (q1 q0)."""
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    qc.ry(-2 * alpha, 0)
    qc.ry(-2 * beta, 1)
    return Statevector(qc).probabilities()


def correlation_sv(alpha: float, beta: float) -> float:
    p = pair_probs(alpha, beta)          # index = 2*photon2_bit + photon1_bit
    return p[0] + p[3] - p[1] - p[2]


# Exact curve
deltas = np.linspace(0.0, np.pi, 25)
defect = max(abs(correlation_sv(d, 0.0) - np.cos(2 * d)) for d in deltas)
assert defect < 1e-12, "E(Delta) must equal cos 2 Delta"
print(f"statevector sweep, 25 angles: max |E - cos 2 Delta| = {defect:.2e}")

# The guide's Preset D anchor: E(0, 22.5 deg) = sqrt2/2
shots = 10_000
c = sample_counts(pair_circuit(0.0, np.pi / 8), shots, SEED + 14)
lab = {qiskit_to_lab(k): v for k, v in c.items()}
e_meas = (lab.get('00', 0) + lab.get('11', 0)
          - lab.get('01', 0) - lab.get('10', 0)) / shots
e_th = np.sqrt(2) / 2
sigma_e = np.sqrt((1 - e_th ** 2) / shots)     # Var(E) = (1 - E^2)/n
assert abs(e_meas - e_th) < 4 * sigma_e
print(f"sampled E(0, 22.5 deg) = {e_meas:.4f}   theory {e_th:.4f} "
      f"(+- {4*sigma_e:.4f} at 4 sigma)")

# Sampled dots + both theory curves (the guide's Preset D overlay)
dot_deltas = np.linspace(0.0, np.pi, 9)
dots = []
for i, d in enumerate(dot_deltas):
    cd = sample_counts(pair_circuit(d, 0.0), 2000, SEED + 200 + i)
    lab_d = {qiskit_to_lab(k): v for k, v in cd.items()}
    dots.append((lab_d.get('00', 0) + lab_d.get('11', 0)
                 - lab_d.get('01', 0) - lab_d.get('10', 0)) / 2000)

grid = np.linspace(0, np.pi, 200)
plt.figure(figsize=(7, 3.2))
plt.plot(grid, np.cos(2 * grid), label="quantum $E = \\cos 2\\Delta$")
plt.plot(grid, np.cos(2 * grid) / 2, "--", label="classical wave $\\cos(2\\Delta)/2$")
plt.plot(dot_deltas, dots, "o", label="sampled (2000 pairs/point)")
plt.xlabel("analyzer difference $\\Delta$ [rad]"); plt.ylabel("$E(\\Delta)$")
plt.title("Pair correlation: the factor-of-2 gap (compare Preset D)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"\nAt matched analyzers: quantum agreement = 100%; classical wave manages")
print(f"E_cl(0) = {np.cos(0.0)/2:.2f} -> only 75% agreement. The data above sit on the")
print("solid curve: no classical wave story survives, and more sophisticated local")
print("models fail elsewhere on the curve (Bell's theorem - cited, not derived here).")

---
## 6 · Optional: The Interferometer on Real Hardware

Everything above ran on a noiseless simulator with structural zeros that were *exactly*
zero. A physical device has a noise floor: fringes lose contrast, "impossible" outcomes
acquire small rates. Running the $\varphi = \pi/3$ interferometer and the perfect-marking
circuit on an IBM backend shows both effects — and the contrast loss *is* a visibility
measurement, connecting directly to $\mathcal{V} = s$: noise is an unread environment
marker.

In [ ]:
"""
Hardware run (optional, off by default): the phi = pi/3 interferometer and the
perfect-marking circuit on an IBM Quantum backend.
Requires: pip install qiskit-ibm-runtime, and a saved IBM Quantum account.
"""
RUN_ON_HARDWARE = False

if not RUN_ON_HARDWARE:
    print("Skipped (RUN_ON_HARDWARE = False).")
    print("Enable it to measure the hardware fringe at phi = pi/3 (ideal: 75/25) and")
    print("the coincidence rate of the marked interferometer - the device noise floor")
    print("that real photonic experiments must engineer against.")
else:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    service = QiskitRuntimeService()             # uses your saved account
    backend = service.least_busy(operational=True, simulator=False)
    print("backend:", backend.name)

    qc_mz = mz_circuit(np.pi / 3)
    qc_mz.measure_all()
    qc_mark = QuantumCircuit(2)
    qc_mark.h(0)
    qc_mark.cx(0, 1)
    qc_mark.measure_all()

    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    job = SamplerV2(mode=backend).run(pm.run([qc_mz, qc_mark]), shots=4096)
    res = job.result()

    mz_counts = res[0].data.meas.get_counts()
    frac_d1 = mz_counts.get('0', 0) / sum(mz_counts.values())
    print(f"hardware D1 fraction at phi = pi/3: {frac_d1:.4f}  (ideal 0.7500)")

    mark_counts = res[1].data.meas.get_counts()
    lab_marks = {qiskit_to_lab(k): v for k, v in mark_counts.items()}
    bad = lab_marks.get('01', 0) + lab_marks.get('10', 0)
    print(f"marked circuit, lab keys: { {k: lab_marks[k] for k in sorted(lab_marks)} }")
    print(f"'impossible' mixed outcomes: {bad} of {sum(mark_counts.values())} "
          "(simulator value was exactly 0 - the difference is the noise floor)")

---
## Summary

| Quantity | Theory | Measured in | Also in the guide's simulator |
|---|---|---|---|
| Beam-splitter split | $1/2,\ 1/2$ | Section 1 | Preset A |
| Coincidences, one photon | $0$ **exactly** | Section 1 (structural) | Preset A counter |
| MZ bright port, $\varphi = 0$ | $1$ **exactly** | Section 2 (structural) | Preset B1 |
| MZ fringe, $\varphi = \pi/3$ | $3/4$ | Section 2 | Preset B1, slider at 60° |
| Circuit ↔ bench relation | $B\Phi B = S\,HP(\varphi{+}\pi)H\,S$ | Section 2, numeric | — |
| Visibility vs marking | $\mathcal{V} = s$ | Section 3 | Preset B2 |
| Perfect-marking mixed keys | $0$ **exactly** | Section 3 (structural) | — |
| Malus at 60° | $1/4$ | Section 4 | Preset C, single 60° |
| Crossed / three polarizers | $0$ / $1/8$ | Section 4 | Preset C |
| Staircase 0/30/60/90 | $27/128$ | Section 4 | Preset C |
| Matched-pair agreement | $1$ **exactly** | Section 5 (structural) | Preset D |
| $E(0°, 22.5°)$ | $\sqrt2/2 \approx 0.707$ | Section 5 | Preset D |
| Classical-wave correlation | $\cos(2\Delta)/2$ — half | Section 5 overlay | Preset D overlay |

**Takeaways.** One circuit family reproduced every number of the optical bench — because
the photon's path and polarization *are* qubits, and the bench devices *are* gates. The
representation change ($B$ vs $H$-circuit) was derived, not glossed: an exact identity
plus a phase relabeling invisible to Z-measurement. The quiet star is again the pair of
distinctions from Lab 5: **Procedure 1 vs Procedure 2** (same statistics, different
post-states — three polarizers transmit *because* projection rewrites the state) and
**distinguishability vs disturbance** ($\mathcal{V} = s$: an unread record kills the
fringes, which is Lab 6's security argument in one identity). And the entangled pairs
close the arc: perfectly correlated, individually random, and separated from every
classical wave story by a measured factor of 2.